In [0]:
spark.conf.set(
    "fs.azure.account.key.adlsstoragertp9.dfs.core.windows.net",
    "QhKofrsyaZYX9oH3BH56HhGubXsbCv/h9hrkjJMXJpDaOpPbYCnXRvh2cvGf8UYo1kyThYW5jaz9+AStplQf8g=="
)

In [0]:
from pyspark.sql.functions import col, sum, avg, count, round, month, year, datediff, rank, desc
from pyspark.sql.window import Window

spark.sql("CREATE CATALOG IF NOT EXISTS ecommerce_gold")
spark.sql("CREATE SCHEMA IF NOT EXISTS ecommerce_gold.sales")

orders    = spark.table("ecommerce_silver.sales.orders")
items     = spark.table("ecommerce_silver.sales.order_items")
payments  = spark.table("ecommerce_silver.sales.payments")
customers = spark.table("ecommerce_silver.sales.customers")
products  = spark.table("ecommerce_silver.sales.products")
sellers   = spark.table("ecommerce_silver.sales.sellers")

print("Silver tables loaded!")

Silver tables loaded!


In [0]:
monthly_revenue = payments \
    .join(orders, "order_id") \
    .withColumn("year", year("order_purchase_timestamp")) \
    .withColumn("month", month("order_purchase_timestamp")) \
    .groupBy("year", "month") \
    .agg(
        round(sum("payment_value"), 2).alias("total_revenue"),
        count("order_id").alias("total_orders"),
        round(avg("payment_value"), 2).alias("avg_order_value")
    ).orderBy("year", "month")

monthly_revenue.write.mode("overwrite").format("delta") \
    .saveAsTable("ecommerce_gold.sales.monthly_revenue")

print(f"Monthly revenue: {monthly_revenue.count()} rows")
display(monthly_revenue)

Monthly revenue: 25 rows


year,month,total_revenue,total_orders,avg_order_value
2016,9,252.24,3,84.08
2016,10,59090.48,342,172.78
2016,12,19.62,1,19.62
2017,1,138488.04,850,162.93
2017,2,291908.01,1886,154.78
2017,3,449863.6,2837,158.57
2017,4,417788.03,2569,162.63
2017,5,592918.82,3943,150.37
2017,6,511276.38,3435,148.84
2017,7,592382.92,4317,137.22


In [0]:
%sql
SELECT * FROM ecommerce_gold.sales.monthly_revenue ORDER BY year, month

year,month,total_revenue,total_orders,avg_order_value
2016,9,252.24,3,84.08
2016,10,59090.48,342,172.78
2016,12,19.62,1,19.62
2017,1,138488.04,850,162.93
2017,2,291908.01,1886,154.78
2017,3,449863.6,2837,158.57
2017,4,417788.03,2569,162.63
2017,5,592918.82,3943,150.37
2017,6,511276.38,3435,148.84
2017,7,592382.92,4317,137.22


In [0]:
%sql
SELECT * FROM ecommerce_gold.sales.top_categories LIMIT 10

product_category_name_english,total_revenue,total_orders,avg_price
health_beauty,1258681.34,9670,130.16
watches_gifts,1205005.68,5991,201.14
bed_bath_table,1036988.68,11115,93.3
sports_leisure,988048.97,8641,114.34
computers_accessories,911954.32,7827,116.51
furniture_decor,729762.49,8334,87.56
cool_stuff,635290.85,3796,167.36
housewares,632248.66,6964,90.79
auto,592720.11,4235,139.96
garden_tools,485256.46,4347,111.63


In [0]:
top_categories = items \
    .join(products, "product_id") \
    .groupBy("product_category_name_english") \
    .agg(
        round(sum("price"), 2).alias("total_revenue"),
        count("order_id").alias("total_orders"),
        round(avg("price"), 2).alias("avg_price")
    ) \
    .filter(col("product_category_name_english").isNotNull()) \
    .orderBy(col("total_revenue").desc())

top_categories.write.mode("overwrite").format("delta") \
    .saveAsTable("ecommerce_gold.sales.top_categories")

print(f"Top categories: {top_categories.count()} rows")
display(top_categories.limit(10))

Top categories: 71 rows


product_category_name_english,total_revenue,total_orders,avg_price
health_beauty,1258681.34,9670,130.16
watches_gifts,1205005.68,5991,201.14
bed_bath_table,1036988.68,11115,93.3
sports_leisure,988048.97,8641,114.34
computers_accessories,911954.32,7827,116.51
furniture_decor,729762.49,8334,87.56
cool_stuff,635290.85,3796,167.36
housewares,632248.66,6964,90.79
auto,592720.11,4235,139.96
garden_tools,485256.46,4347,111.63


In [0]:
delivery_perf = orders \
    .filter(col("order_delivered_customer_date").isNotNull()) \
    .withColumn("delivery_days",
        datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))) \
    .join(customers, "customer_id") \
    .groupBy("customer_state") \
    .agg(
        round(avg("delivery_days"), 1).alias("avg_delivery_days"),
        count("order_id").alias("total_orders"),
        round(sum(
            (col("order_delivered_customer_date") > col("order_estimated_delivery_date")).cast("int")
        ) / count("order_id") * 100, 1).alias("late_delivery_pct")
    ) \
    .orderBy("avg_delivery_days")

delivery_perf.write.mode("overwrite").format("delta") \
    .saveAsTable("ecommerce_gold.sales.delivery_performance")

print(f"Delivery performance: {delivery_perf.count()} rows")
display(delivery_perf)

Delivery performance: 27 rows


customer_state,avg_delivery_days,total_orders,late_delivery_pct
SP,8.7,40495,5.9
PR,11.9,4923,5.0
MG,11.9,11355,5.6
DF,12.9,2080,7.1
SC,14.9,3547,9.8
RS,15.2,5344,7.1
RJ,15.2,12353,13.5
GO,15.5,1957,8.2
MS,15.5,701,11.6
ES,15.7,1995,12.2


In [0]:
payment_methods = payments \
    .groupBy("payment_type") \
    .agg(
        count("order_id").alias("total_transactions"),
        round(sum("payment_value"), 2).alias("total_value"),
        round(avg("payment_value"), 2).alias("avg_value"),
        round(avg("payment_installments"), 1).alias("avg_installments")
    ) \
    .orderBy(col("total_transactions").desc())

payment_methods.write.mode("overwrite").format("delta") \
    .saveAsTable("ecommerce_gold.sales.payment_methods")

print(f"Payment methods: {payment_methods.count()} rows")
display(payment_methods)

Payment methods: 4 rows


payment_type,total_transactions,total_value,avg_value,avg_installments
credit_card,76795,1.254208419E7,163.32,3.5
boleto,19784,2869361.27,145.03,1.0
voucher,5769,379436.87,65.77,1.0
debit_card,1529,217989.79,142.57,1.0


In [0]:
top_sellers = items \
    .join(sellers, "seller_id") \
    .groupBy("seller_id", "seller_state", "seller_city") \
    .agg(
        round(sum("price"), 2).alias("total_revenue"),
        count("order_id").alias("total_orders"),
        round(avg("price"), 2).alias("avg_order_value")
    ) \
    .orderBy(col("total_revenue").desc())

top_sellers.write.mode("overwrite").format("delta") \
    .saveAsTable("ecommerce_gold.sales.top_sellers")

print(f"Top sellers: {top_sellers.count()} rows")
display(top_sellers.limit(10))

Top sellers: 3095 rows


seller_id,seller_state,seller_city,total_revenue,total_orders,avg_order_value
4869f7a5dfa277a7dca6462dcf3b52b2,SP,guariba,229472.63,1156,198.51
53243585a1d6dc2643021fd1853d8905,BA,lauro de freitas,222776.05,410,543.36
4a3ca9315b744ce9f8e9374361493884,SP,ibitinga,200472.92,1987,100.89
fa1c13f2614d7b5c4749cbc52fecda94,SP,sumare,194042.03,586,331.13
7c67e1448b00f6e969d365cea6b010ab,SP,itaquaquecetuba,187923.89,1364,137.77
7e93a43ef30c4f03f38b393420bc753a,SP,barueri,176431.87,340,518.92
da8622b14eb17ae2831f4ac5b9dab84a,SP,piracicaba,160236.57,1551,103.31
7a67c85e85bb2ce8582c35f2203ad736,SP,sao paulo,141745.53,1171,121.05
1025f0e2d44d7041d6cf58b6550e0bfa,SP,sao paulo,138968.55,1428,97.32
955fee9216a65b617aa5c0531780ce60,SP,sao paulo,135171.7,1499,90.17


In [0]:
gold_tables = [
    "monthly_revenue", "top_categories",
    "delivery_performance", "payment_methods", "top_sellers"
]

print("=== Gold Layer Summary ===")
for t in gold_tables:
    count = spark.table(f"ecommerce_gold.sales.{t}").count()
    print(f"ecommerce_gold.sales.{t}: {count} rows")

=== Gold Layer Summary ===
ecommerce_gold.sales.monthly_revenue: 25 rows
ecommerce_gold.sales.top_categories: 71 rows
ecommerce_gold.sales.delivery_performance: 27 rows
ecommerce_gold.sales.payment_methods: 4 rows
ecommerce_gold.sales.top_sellers: 3095 rows
